# 00b — Quarantine: Affective Lexicons (Warriner, NRC-VAD)
*these are **partial-coverage** dictionary methods (~23% of words in this spoken-transcript data; out-of-vocabulary words are skipped), which conflicts with the full-text-coverage criterion.

This notebook is **self-contained** and is now the **sole producer** of the lexicon outputs that the rest of the pipeline may read (Step 1 reads the Warriner/NRC-VAD `_val` baselines as reference models). It regenerates, into the organized `results/` tree:
- `results/scored/00b__reflection_affective_lexicons.csv` — per-reflection VAD scores
- `results/baselines/character_vectors_simple_{Warriner,NRC_VAD}_{vad,val}.csv` — baselines
- `results/profiles/` + `results/deltas/` — lexicon VAD profiles (for the later RSA/brain step)

Kept documented in case dimensional valence/arousal/dominance is wanted later. The current-standard full-coverage replacement would be a transformer VAD regressor trained on EmoBank (Park et al. 2021) or GPT-4o annotation (Rathje et al. 2024, PNAS) — **not** a newer lexicon (even NRC-VAD v2, 2025, is still word-lookup). I used claude to move this code that used to exist in nobteook 00 to its own notbeook named 00b. it largely wrote the cell prefaces

In [1]:
# Standalone setup: load the per-reflection table (Raw_Text) produced by Step 0.1/0.2.
import pandas as pd, numpy as np
from pathlib import Path

df = pd.read_csv("results/scored/00__reflection_sentiment.csv")[["Participant","Run","Character","Raw_Text"]].copy()
df["Raw_Text"] = df["Raw_Text"].fillna("")
print("loaded reflections:", df.shape)

loaded reflections: (1272, 4)


## 0.2b Field-standard affective tools used in neuroscience / cog-sci
 Cognitive neuro uses  **human-rated affective lexicons** (valence/arousal/dominance) and, for brain-encoding, **contextual embeddings**.

- **VAD lexicons** (Warriner, NRC-VAD) output continuous **valence / arousal / dominance** per word, averaged over the reflection. Valence is one *bipolar* axis, not separate pos/neg
- **GPT-2 / USE** are high-dim *semantic embeddings* (brain-encoding angle), a different feature class from sentiment — they don't go in the pos/neg `MODELS` dict. **USE is what Jin's paper used!!** (no dedicated sentiment classifier in that paper).

STUDIES: Warriner, Kuperman & Brysbaert (2013); Mohammad (2018, NRC-VAD); Cer et al. (2018, USE); Schrimpf et al. (2021) / Goldstein et al. (2022) for GPT-2 as best brain-aligned LM.

**Download (free, not bundled):** `Ratings_Warriner_et_al.csv` (suppl. to 2013 paper); `NRC-VAD-Lexicon.csv` (saifmohammad.com).

In [2]:
# Affective lexicons: Warriner VAD + NRC-VAD (word-averaged over the reflection)
# Warriner: V/A/D on 1-9 (5=neutral).  NRC-VAD: V/A/D on 0-1 (0.5=neutral).
# Different scales -> keep separate; z-score later if you ever fuse them (your rule).
import re as _re

def _tokens(text):
    return _re.findall(r"[a-z']+", str(text).lower())

WARRINER_CSV = "data/lexicons/Ratings_Warriner_et_al.csv"   #columns= Word, V.Mean.Sum, A.Mean.Sum, D.Mean.Sum
try:
    _w = pd.read_csv(WARRINER_CSV).set_index("Word")
    _wv, _wa, _wd = _w["V.Mean.Sum"], _w["A.Mean.Sum"], _w["D.Mean.Sum"]
    def score_warriner(text):
        toks = [t for t in _tokens(text) if t in _wv.index]
        if not toks: return {"val": np.nan, "aro": np.nan, "dom": np.nan, "coverage": 0.0}
        return {"val": float(_wv.loc[toks].mean()), "aro": float(_wa.loc[toks].mean()),
                "dom": float(_wd.loc[toks].mean()), "coverage": len(toks)/max(len(_tokens(text)),1)}
except Exception as e:
    print(f"[Warriner not loaded: {e}] -> place {WARRINER_CSV} alongside the notebook")
    def score_warriner(text): return {"val": np.nan, "aro": np.nan, "dom": np.nan, "coverage": np.nan}

NRCVAD_TXT = "data/lexicons/NRC-VAD-Lexicon.txt" 
try:
    _peek = pd.read_csv(NRCVAD_TXT, sep="\t", nrows=1, header=None)
    _has_header = str(_peek.iloc[0, 1]).replace(".", "").replace("-", "").isalpha()

    if _has_header:
        _n = pd.read_csv(NRCVAD_TXT, sep="\t")
        _n.columns = [c.strip().lower() for c in _n.columns]
        _n = _n.rename(columns={_n.columns[0]: "word"}).set_index("word")
    else:
        _n = pd.read_csv(NRCVAD_TXT, sep="\t", header=None,
                         names=["word", "valence", "arousal", "dominance"]).set_index("word")

    _n[["valence", "arousal", "dominance"]] = _n[["valence", "arousal", "dominance"]].apply(pd.to_numeric) #check if numeric, snaity check
    print("NRC-VAD loaded:", _n.shape, "| columns:", list(_n.columns))

    def score_nrc_vad(text):
        toks = [t for t in _tokens(text) if t in _n.index]
        if not toks: return {"val": np.nan, "aro": np.nan, "dom": np.nan}
        sub = _n.loc[toks]
        return {"val": float(sub["valence"].mean()), "aro": float(sub["arousal"].mean()),
                "dom": float(sub["dominance"].mean())}
except Exception as e:
    print(f"[NRC-VAD not loaded: {e}] -> place {NRCVAD_TXT} alongside the notebook")
    def score_nrc_vad(text): return {"val": np.nan, "aro": np.nan, "dom": np.nan}

NRC-VAD loaded: (54801, 3) | columns: ['valence', 'arousal', 'dominance']


In [3]:
#score every reflection with the VAD lexicons making their OWN table (not the pos/neg MODELS dict)
LEX = {"Warriner": score_warriner, "NRC_VAD": score_nrc_vad}
lex_scored = df[["Participant","Run","Character"]].copy()
for name, fn in LEX.items():
    recs = pd.DataFrame(list(df["Raw_Text"].map(fn)))
    recs.columns = [f"{name}_{c}" for c in recs.columns]
    lex_scored = pd.concat([lex_scored, recs], axis=1)
Path("results/scored").mkdir(parents=True, exist_ok=True)
lex_scored.to_csv("results/scored/00b__reflection_affective_lexicons.csv", index=False)
print("wrote reflection_affective_lexicons.csv", lex_scored.shape); lex_scored.head()

wrote reflection_affective_lexicons.csv (1272, 10)


,Participant,Run,Character,Warriner_val,Warriner_aro,Warriner_dom,Warriner_coverage,NRC_VAD_val,NRC_VAD_aro,NRC_VAD_dom
0,sub-1001,1,jack,6.880000,4.320000,5.424000,0.151515,0.181333,-0.041619,0.027810
1,sub-1001,1,randall,5.928000,4.045333,5.499333,0.267857,0.221129,-0.125258,0.019581
2,sub-1001,1,kevin,5.960000,4.028000,5.387000,0.256410,0.109760,-0.050800,-0.004920
3,sub-1001,1,kate,5.461429,4.255714,5.565714,0.233333,0.067441,-0.048029,0.037588
4,sub-1001,2,kate,5.882308,3.793077,5.754615,0.317073,0.200083,-0.148833,0.086667


In [4]:
### 0.2b-report Lexicon coverage & NaNs
#Lexicons score only the words they contain, then average. A reflection with **zero** in-lexicon words yields **NaN** (no reading — not neutral). We KEEP NaNs (no imputation) and report coverage, so you know how much of each reflection the lexicon actually saw.
#Coverage / NaN report for the affective lexicons (we keep NaN, never impute)
for lex in ["Warriner", "NRC_VAD"]:
    valcol, covcol = f"{lex}_val", f"{lex}_coverage"
    if valcol not in lex_scored.columns: continue
    n_nan = lex_scored[valcol].isna().sum()
    cov = lex_scored[covcol] if covcol in lex_scored.columns else None
    print(f"{lex}:")
    print(f"  reflections with NO in-lexicon words (NaN valence): {n_nan} / {len(lex_scored)}")
    if cov is not None:
        print(f"  mean word coverage per reflection: {cov.mean():.1%}  "
              f"(min {cov.min():.1%}, max {cov.max():.1%})")
    print()


Warriner:
  reflections with NO in-lexicon words (NaN valence): 16 / 1272
  mean word coverage per reflection: 23.6%  (min 0.0%, max 50.0%)

NRC_VAD:
  reflections with NO in-lexicon words (NaN valence): 1 / 1272



## 0.3b Lexicon baselines
Two baseline files per lexicon: full **[valence, arousal, dominance]** (`_vad`) and **valence-only** (`_val`, apples-to-apples with the classifiers' pos/neg). A different feature type from pos/neg, so kept as their own baseline files. Step 1 reads the `_val` versions.

In [5]:
Path("results/baselines").mkdir(parents=True, exist_ok=True)
for lex in ["Warriner", "NRC_VAD"]:
    cols = [c for c in [f"{lex}_val", f"{lex}_aro", f"{lex}_dom"] if c in lex_scored.columns]
    if not cols:
        print(f"[skip {lex}: no columns]"); continue
    base = lex_scored[["Participant","Run","Character"] + cols]
    base.to_csv(f"results/baselines/00b__character_vectors_simple_{lex}_vad.csv", index=False)
    print(f"wrote character_vectors_simple_{lex}_vad.csv  {base.shape}")
    if f"{lex}_val" in lex_scored.columns:
        valb = (lex_scored[["Participant","Run","Character", f"{lex}_val"]]
                .rename(columns={f"{lex}_val": "valence"}))
        valb.to_csv(f"results/baselines/00b__character_vectors_simple_{lex}_val.csv", index=False)
        print(f"wrote character_vectors_simple_{lex}_val.csv  {valb.shape}")

wrote character_vectors_simple_Warriner_vad.csv  (1272, 6)
wrote character_vectors_simple_Warriner_val.csv  (1272, 4)
wrote character_vectors_simple_NRC_VAD_vad.csv  (1272, 6)
wrote character_vectors_simple_NRC_VAD_val.csv  (1272, 4)


## 0.5b Lexicon profiles (moved here from the old Step-0 notebook)
Same three views as the classifier profiles — raw **levels** (primary), **mean-centered** (shape), and **deltas** (run-to-run change) — built on the lexicons' VAD features. Saved to `results/profiles/` and `results/deltas/`, matching the classifier profile layout. These feed the later RSA/brain step, not Step 1.

In [6]:
Path("results/profiles").mkdir(parents=True, exist_ok=True)
Path("results/deltas").mkdir(parents=True, exist_ok=True)

def build_profiles(score_df, feature_cols, method):
    levels = (score_df[["Participant","Run","Character"] + feature_cols]
              .dropna(subset=feature_cols, how="all")
              .sort_values(["Participant","Character","Run"]))
    levels.to_csv(f"results/profiles/00b__profiles_{method}.csv", index=False)
    centered = levels.copy()
    centered[feature_cols] = (centered.groupby(["Participant","Character"])[feature_cols]
                              .transform(lambda x: x - x.mean()))
    centered.to_csv(f"results/profiles/00b__profiles_{method}_centered.csv", index=False)
    delta_rows = []
    for (p, ch), g in levels.groupby(["Participant","Character"]):
        g = g.sort_values("Run"); runs = g["Run"].tolist()
        X = g[feature_cols].to_numpy(dtype=float)
        for k in range(1, len(runs)):
            row = {"Participant": p, "Character": ch, "Run_from": runs[k-1], "Run_to": runs[k]}
            row.update({f"d_{c}": (X[k]-X[k-1])[j] for j, c in enumerate(feature_cols)})
            delta_rows.append(row)
    pd.DataFrame(delta_rows).to_csv(f"results/deltas/00b__delta_profiles_{method}.csv", index=False)
    print(f"{method:14s} levels {levels.shape}  centered {centered.shape}")

for lex in ["Warriner", "NRC_VAD"]:
    cols = [c for c in [f"{lex}_val", f"{lex}_aro", f"{lex}_dom"] if c in lex_scored.columns]
    if cols: build_profiles(lex_scored, cols, f"{lex}_vad")

Warriner_vad   levels (1256, 6)  centered (1256, 6)
NRC_VAD_vad    levels (1271, 6)  centered (1271, 6)


## 0.2c EmoBank VAD regressor — modern, full-coverage replacement for the lexicons
Warriner/NRC-VAD are **word lookups** (~23% coverage; a reflection with no in-lexicon words gets NaN). A regressor **trained on EmoBank** predicts continuous valence for **every** reflection (full coverage, no NaN) — the standard modern fix (Park et al. 2021). Trained and run **fully locally**: TF-IDF + Ridge, no GPU, no API. (For a stronger version, swap TF-IDF for frozen transformer embeddings.)

**Download once** (free): EmoBank `corpus/emobank.csv` from https://github.com/JULIELab/EmoBank — place it at `data/lexicons/emobank.csv`. Columns: `id, split, V, A, D, text` (V/A/D on 1-5, 3 = neutral). The cell skips gracefully if the file isn't there.

**Reviewer note:** this is a TF-IDF + Ridge *proxy* for an EmoBank-trained regressor, chosen so it runs locally with no GPU. It is a bag-of-words model, weaker than a transformer VAD regressor; label it as such if reported.

In [7]:
from pathlib import Path
EMOBANK_CSV = Path("data/lexicons/emobank.csv")   # download from github.com/JULIELab/EmoBank
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.model_selection import cross_val_score

    eb = pd.read_csv(EMOBANK_CSV)                  # id, split, V, A, D, text
    reg = {}
    for dim in ["V", "A", "D"]:
        m = make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2)), Ridge(alpha=1.0))
        r2 = cross_val_score(m, eb["text"].astype(str), eb[dim].astype(float), cv=5, scoring="r2").mean()
        m.fit(eb["text"].astype(str), eb[dim].astype(float))
        reg[dim] = m
        print(f"EmoBank {dim} regressor  CV R^2 = {r2:.3f}")

    emob = df[["Participant", "Run", "Character"]].copy()
    txt = df["Raw_Text"].astype(str)
    emob["EmoBank_val"] = reg["V"].predict(txt)
    emob["EmoBank_aro"] = reg["A"].predict(txt)
    emob["EmoBank_dom"] = reg["D"].predict(txt)

    Path("results/baselines").mkdir(parents=True, exist_ok=True)
    emob.to_csv("results/baselines/00b__character_vectors_simple_EmoBank_vad.csv", index=False)
    (emob[["Participant", "Run", "Character", "EmoBank_val"]]
        .rename(columns={"EmoBank_val": "valence"})
        .to_csv("results/baselines/00b__character_vectors_simple_EmoBank_val.csv", index=False))
    print("EmoBank full-coverage valence written:", emob.shape,
          "| NaNs:", int(emob["EmoBank_val"].isna().sum()), "(lexicons have NaNs; this has none)")
except FileNotFoundError:
    print(f"[EmoBank skipped: place emobank.csv at {EMOBANK_CSV} -- see github.com/JULIELab/EmoBank]")

[EmoBank skipped: place emobank.csv at data/lexicons/emobank.csv -- see github.com/JULIELab/EmoBank]
